In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from src.pipelines.features.validate import load_raw_data

rides = load_raw_data(year=2025)
rides

,pickup_datetime,pickup_location_id
0,2025-01-01 00:18:38,229
1,2025-01-01 00:32:40,236
2,2025-01-01 00:44:04,141
3,2025-01-01 00:14:27,244
4,2025-01-01 00:21:34,244
...,...,...
4305001,2025-12-31 23:20:55,216
4305002,2025-12-31 23:08:47,22
4305003,2025-12-31 23:29:04,128
4305004,2025-12-31 23:25:12,114


In [4]:
from src.pipelines.features.transform_raw_to_ts import transform_raw_data_into_ts_data

ts_data = transform_raw_data_into_ts_data(rides)
ts_data

100%|██████████| 265/265 [00:01<00:00, 189.94it/s]


,pickup_hour,rides,pickup_location_id
0,2025-01-01 00:00:00,0,1
1,2025-01-01 01:00:00,0,1
2,2025-01-01 02:00:00,0,1
3,2025-01-01 03:00:00,0,1
4,2025-01-01 04:00:00,0,1
...,...,...,...
2321395,2025-12-31 19:00:00,4,265
2321396,2025-12-31 20:00:00,3,265
2321397,2025-12-31 21:00:00,6,265
2321398,2025-12-31 22:00:00,3,265


In [8]:
from src.pipelines.features.transform_ts_to_features import transform_ts_data_into_features_and_target

features, target = transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len = 24*28*1, #one month
    step_size = 24
)

print(f'{features.shape=}')
print(f'{target.shape=}')

100%|██████████| 265/265 [00:12<00:00, 21.56it/s]

features.shape=(89305, 674)
target.shape=(89305,)


In [9]:
features.head(5)

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,pickup_location_id
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-01-29,1
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-01-30,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,2025-01-31,1
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2025-02-01,1
4,0.0,1.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,...,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-02-02,1


In [11]:
tabular_data = features
tabular_data['target_rides_next_hour'] = target

from src.utils.paths import TRANSFORMED_DATA_DIR

tabular_data.to_parquet(TRANSFORMED_DATA_DIR / "tabular_data.parquet")